### 1: Data Ingestion & State Distribution Inspection

In [ ]:
import os
import sys
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Add src folder to Python module search path
src_dir = os.path.abspath("src")
if src_dir not in sys.path:
    sys.path.append(src_dir)

from helpers import (
    load_rollforming_df,
    split_data_statewise,
    train_model_full,
    evaluate_model,
    set_seeds
)
from model import ConvRegressionModel

# Hardware selection
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Active Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Define path to data inside src
data_path = os.path.join(src_dir, "asym_data.csv")
assert os.path.exists(data_path), f"File {data_path} not found."
print(f"Data file confirmed at: {data_path}")
df = load_rollforming_df(data_path)

unique_ids = df['id'].unique()
n_profiles = len(unique_ids)
steps_per_profile = df.groupby('id').size().unique()

force_cols = [c for c in df.columns if 'kraft' in c.lower()]
target_cols = ['X_opt - X_ist', 'Y_opt - Y_ist', 'Phi_opt - Phi_ist']

print(f"Total rows: {len(df)}")
print(f"Number of distinct profiles (IDs): {n_profiles}")
print(f"Time steps per profile: {steps_per_profile[0] if len(steps_per_profile) == 1 else steps_per_profile}")
print(f"Force Channels ({len(force_cols)}): {force_cols}")
print(f"Target Deviations ({len(target_cols)}): {target_cols}")

df[target_cols].describe().T[['mean', 'std', 'min', 'max']]

### 2: Statewise Partitioning

In [ ]:
WINDOW_SIZE = 10
SEED = 42

(
    (X_train, Y_train, C_train),
    (X_val, Y_val, C_val),
    (X_test, Y_test, C_test)
) = split_data_statewise(
    file_path=data_path,
    window_size=WINDOW_SIZE,
    test_ratio=0.20,
    val_ratio=0.10,
    seed=SEED,
    class_column='id',
    only_forces=True
)

train_ids = np.unique(C_train)
val_ids = np.unique(C_val)
test_ids = np.unique(C_test)

print(f"Profile ID counts -> Train: {len(train_ids)} | Val: {len(val_ids)} | Test: {len(test_ids)}")
print(f"Window tensor shapes:")
print(f"  Train: X = {X_train.shape}, Y = {Y_train.shape}")
print(f"  Val:   X = {X_val.shape},   Y = {Y_val.shape}")
print(f"  Test:  X = {X_test.shape},  Y = {Y_test.shape}")

# Verify zero leakage across sets
overlap = set(train_ids).intersection(set(test_ids)) | set(val_ids).intersection(set(test_ids))
assert len(overlap) == 0, f"Critical leakage detected: {overlap}"
print("Statewise isolation check: PASSED (Zero data leakage).")

### 3: Model Architecture Verification

In [ ]:
n_feats = X_train.shape[2]
test_model = ConvRegressionModel(window_size=WINDOW_SIZE, num_features=n_feats).to(device)

dummy_batch = torch.randn(4, WINDOW_SIZE, n_feats).to(device)
with torch.no_grad():
    dummy_out = test_model(dummy_batch)

print(f"Input batch shape:  {dummy_batch.shape}")
print(f"Output shape:       {dummy_out.shape} -> Expected: (4, 3)")
assert dummy_out.shape == (4, 3), "Dimension mismatch in output tensor."

### 4: Model Training with Validation Checkpointing

In [ ]:
EPOCHS = 30
BATCH_SIZE = 32
LR = 0.001

model, scaler_X, scaler_y, dev = train_model_full(
    F_train=X_train,
    Y_train=Y_train,
    seed=SEED,
    F_val=X_val,
    Y_val=Y_val,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR
)

print("Training finished. Optimal weights loaded.")

### 5: Test Set Evaluation & Metric Analysis

In [ ]:
mae, trues, preds = evaluate_model(model, scaler_X, scaler_y, X_test, Y_test, dev)

targets = ['X_opt - X_ist', 'Y_opt - Y_ist', 'Phi_opt - Phi_ist']
units = ['mm', 'mm', '°']

print("\n--- Test Set MAE (Single Seed) ---")
for t, u, m in zip(targets, units, mae):
    print(f"  {t:20s}: {m:.4f} {u}")

# Scatter plot: True vs Predicted
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, ax in enumerate(axes):
    ax.scatter(trues[:, i], preds[:, i], alpha=0.3, edgecolors='none', color='#1f77b4')
    min_val = min(trues[:, i].min(), preds[:, i].min())
    max_val = max(trues[:, i].max(), preds[:, i].max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=1.5, label='Ideal')
    ax.set_title(f"{targets[i]} ({units[i]})")
    ax.set_xlabel("True Value")
    ax.set_ylabel("Predicted Value")
    ax.legend()
    ax.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

### 6: Multi-Seed Robustness Validation

In [ ]:
seeds = [101, 202, 303, 404, 505, 606, 707, 808, 909, 1001]
results = []

for s in seeds:
    # Statewise split per seed
    (X_tr, Y_tr, _), (X_va, Y_va, _), (X_te, Y_te, _) = split_data_statewise(
        file_path=data_path,
        window_size=WINDOW_SIZE,
        test_ratio=0.20,
        val_ratio=0.10,
        seed=s,
        only_forces=True
    )
    
    # Train and evaluate
    m, sX, sY, d = train_model_full(
        F_train=X_tr,
        Y_train=Y_tr,
        seed=s,
        F_val=X_va,
        Y_val=Y_va,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LR
    )
    s_mae, _, _ = evaluate_model(m, sX, sY, X_te, Y_te, d)
    results.append(s_mae)
    print(f"Seed {s:4d} -> MAE: X={s_mae[0]:.4f} mm | Y={s_mae[1]:.4f} mm | Phi={s_mae[2]:.4f}°")

results = np.array(results)
mean_maes = np.mean(results, axis=0)
std_maes = np.std(results, axis=0)

summary_df = pd.DataFrame({
    'Target Deviation': targets,
    'Unit': units,
    'Mean MAE': [f"{m:.4f}" for m in mean_maes],
    'Std MAE': [f"± {s:.4f}" for s in std_maes]
})

print("\n" + "=" * 50)
print("FINAL BENCHMARK: 10-SEED STATEWISE VALIDATION")
print("=" * 50)
summary_df

### 7. Results to CSV

In [ ]:
# ==========================================
# Export Benchmark & Comparison Tables to CSV
# ==========================================
import os
import numpy as np
import pandas as pd

# Define output directory and ensure it exists
output_dir = os.path.join(os.getcwd(), "results")
os.makedirs(output_dir, exist_ok=True)

# 1. Standard Benchmark Table (Zero-Shot IDs)
benchmark_csv_path = os.path.join(output_dir, "rollforming_benchmark_results.csv")
export_df = pd.DataFrame({
    'Subset': ['Zero-Shot (IDs)'],
    'X_opt - X_ist [mm]': [f"{mean_maes[0]:.4f} ± {std_maes[0]:.4f}"],
    'Y_opt - Y_ist [mm]': [f"{mean_maes[1]:.4f} ± {std_maes[1]:.4f}"],
    'Phi_opt - Phi_ist [°]': [f"{mean_maes[2]:.4f} ± {std_maes[2]:.4f}"]
})
export_df.to_csv(benchmark_csv_path, index=False, sep=',', encoding='utf-8')
print(f"Standard benchmark table saved to: {benchmark_csv_path}")

# 2. Comprehensive Comparison Table (Dataset Baseline vs Model Performance)
comparison_rows = []
for i, col in enumerate(targets):
    col_min = df[col].min()
    col_max = df[col].max()
    col_range = col_max - col_min
    col_std = df[col].std()
    
    # Naive baseline: MAE obtained by always predicting the dataset mean
    dummy_mae = np.mean(np.abs(df[col] - df[col].mean()))
    
    # Model metrics
    m_mae = mean_maes[i]
    m_std = std_maes[i]
    
    # Relative percentage metrics
    err_to_range_pct = (m_mae / col_range) * 100
    err_reduction_pct = ((dummy_mae - m_mae) / dummy_mae) * 100
    
    comparison_rows.append({
        'Target Deviation': f"{col} [{units[i]}]",
        'Dataset Range [Min, Max]': f"[{col_min:.2f}, {col_max:.2f}]",
        'Dataset Std': f"{col_std:.4f}",
        'Naive Baseline MAE': f"{dummy_mae:.4f}",
        'Model MAE (Zero-Shot)': f"{m_mae:.4f} ± {m_std:.4f}",
        'Error / Range [%]': f"{err_to_range_pct:.2f}%",
        'Error Reduction [%]': f"{err_reduction_pct:.2f}%"
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_csv_path = os.path.join(output_dir, "rollforming_comparison_baseline.csv")
comparison_df.to_csv(comparison_csv_path, index=False, sep=',', encoding='utf-8')
print(f"Full baseline comparison table saved to: {comparison_csv_path}\n")

# Display formatted comparison table in notebook output
comparison_df